# 05_build_bm25_index_chunked.ipynb

Build BM25 index in chunks to avoid loading the whole corpus into RAM.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import gc
import pickle
from pathlib import Path

import pandas as pd
from rank_bm25 import BM25Okapi


In [2]:
DATA_PATH = (
    "../data/processed/comments_clean.parquet"
)

INDEX_DIR = Path(
    "../data/indexes/product_comments_bm25"
)

CHUNK_DIR = INDEX_DIR / "chunks"

CHUNK_SIZE = 500000


CHUNK_DIR.mkdir(
    parents=True,
    exist_ok=True
)


In [3]:
comments = pd.read_parquet(
    DATA_PATH
)

comments.head()


,id,title,body,created_at,rate,recommendation_status,is_buyer,product_id,advantages,disadvantages,likes,dislikes,seller_title,seller_code,true_to_size_rate,created_at_gregorian
0,14144758,هری پاتر,عالیه مخصوصا طرحش عکس مجموعه هری پاترم رو میزا...,10 آذر 1399,5.0,recommended,True,1075274,NaN,NaN,1136,8,بیگای استودیو,5AADE,NaN,2020-11-30
1,41782279,روغن ریش,توجه فرمایید که عکس اول واسه ۲ هفته پیشه و عکس...,21 آبان 1401,3.0,recommended,True,6081008,NaN,NaN,896,79,گراندو بیوتی,6XN5S,NaN,2022-11-12
2,49569443,تقلبی,متاسفانه از سال ۹۶ مشتری دیجیکالا هستم.بالای ۴...,3 خرداد 1402,1.0,not_recommended,True,10545754,NaN,NaN,625,60,کافه سرگرمی,AXVST,NaN,2023-05-24
3,43932524,یک نظر بی اغراق!,تصمیم خرید کنسول برای منِ 32 ساله با هزینه شخص...,15 دی 1401,5.0,recommended,True,4153832,NaN,NaN,524,51,گروه آروند,5AD65,NaN,2023-01-05
4,21396693,NaN,اول لاک معمولی میزدم و بعد از خشک شدن کامل این...,9 خرداد 1400,3.0,no_idea,True,2185657,NaN,NaN,450,2,تاباتا,C93UM,NaN,2021-05-30


In [4]:
from src.rag.preprocessing.processor import TextProcessor
from src.rag.utils.text import build_comment_text


processor = TextProcessor()


comments["search_text"] = (
    comments
    .apply(build_comment_text, axis=1)
    .apply(processor.process)
)


documents = comments[
    [
        "id",
        "product_id",
        "body",
        "rate",
        "search_text"
    ]
].copy()


print(
    "Documents:",
    len(documents)
)


Documents: 6153060


In [5]:
for start in range(
    0,
    len(documents),
    CHUNK_SIZE
):

    end = min(
        start + CHUNK_SIZE,
        len(documents)
    )


    print(
        f"Building BM25 chunk {start}:{end}"
    )


    chunk = documents.iloc[
        start:end
    ]


    tokenized_docs = [
        text.split()
        for text in chunk["search_text"]
    ]


    bm25 = BM25Okapi(
        tokenized_docs
    )


    with open(
        CHUNK_DIR / f"chunk_{start}.pkl",
        "wb"
    ) as f:

        pickle.dump(
            bm25,
            f
        )


    del chunk
    del tokenized_docs
    del bm25

    gc.collect()


print("BM25 chunks saved")


Building BM25 chunk 0:500000
Building BM25 chunk 500000:1000000
Building BM25 chunk 1000000:1500000
Building BM25 chunk 1500000:2000000
Building BM25 chunk 2000000:2500000
Building BM25 chunk 2500000:3000000
Building BM25 chunk 3000000:3500000
Building BM25 chunk 3500000:4000000
Building BM25 chunk 4000000:4500000
Building BM25 chunk 4500000:5000000
Building BM25 chunk 5000000:5500000
Building BM25 chunk 5500000:6000000
Building BM25 chunk 6000000:6153060
BM25 chunks saved


In [6]:
documents.to_parquet(
    INDEX_DIR / "metadata.parquet",
    index=False
)

print(
    "Metadata saved"
)


Metadata saved


In [7]:
import glob

files = sorted(
    glob.glob(
        str(CHUNK_DIR / "*.pkl")
    )
)

print(
    "Number of chunks:",
    len(files)
)

files[:5]


Number of chunks: 13


['../data/indexes/product_comments_bm25/chunks/chunk_0.pkl',
 '../data/indexes/product_comments_bm25/chunks/chunk_1000000.pkl',
 '../data/indexes/product_comments_bm25/chunks/chunk_1500000.pkl',
 '../data/indexes/product_comments_bm25/chunks/chunk_2000000.pkl',
 '../data/indexes/product_comments_bm25/chunks/chunk_2500000.pkl']